# PILATES Consist Handoff Starter

Minimal HPC notebook starter for:
1. opening an archived run DB,
2. ingesting a few output files,
3. exporting a scenario shard DB,
4. exporting modified ActivitySim trips/persons inputs.


In [ ]:
from pathlib import Path
import pandas as pd

from pilates_consist_analysis import open_run, ArtifactIngestSpec, TableTransformSpec

ARCHIVE_RUN_DIR = Path('/path/to/archived/run')
PROJECT_ROOT = Path('/Users/zaneedell/git/PILATES')
OUTPUT_DIR = Path('/scratch/$USER/pilates-analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_RUN_DIR, OUTPUT_DIR


## 1) Open Session + Basic Health Checks

Use `strict_tagging=True` once you trust your archives are consistently tagged.


In [ ]:
session = open_run(
    ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    strict_tagging=False,
)

print('Tagging warnings:', session.tagging_warnings)
display(session.inspect_db())
display(session.inspect_run_tagging(strict=False))


## 2) Ingest Selected Files Into the DB

Edit the file paths/keys below for the outputs you want hot-loaded.


In [ ]:
# Option A: direct paths
# artifact_specs = [
#     ArtifactIngestSpec(path='/scratch/path/to/trips.csv.gz', key='trips', artifact_family='trips', driver='csv'),
#     ArtifactIngestSpec(path='/scratch/path/to/persons.csv.gz', key='persons', artifact_family='persons', driver='csv'),
# ]

# Option B (recommended): resolve existing tracked artifacts by run/key
artifact_specs = [
    ArtifactIngestSpec(source_run_id='RUN_ID_FOR_ASIM', source_key='trips'),
    ArtifactIngestSpec(source_run_id='RUN_ID_FOR_ASIM', source_key='persons'),
]

# Discover available keys if needed:
# session.list_run_artifacts(run_id='RUN_ID_FOR_ASIM', direction='output')

# Uncomment to run:
# ingest_payload = session.ingest_artifacts(
#     artifact_specs,
#     scenario_id='baseline-2030',
#     year=2030,
#     iteration=3,
#     model='analysis_ingest',
# )
# ingest_payload


## 3) Export Scenario Shard + Query Outputs

Use these as the first reproducible handoff artifacts from the HPC node.


In [ ]:
# 3a) Export standalone DB for one scenario
# shard_payload = session.export_scenario_db(
#     scenario_id='baseline-2030',
#     out_path=OUTPUT_DIR / 'baseline_2030.duckdb',
# )
# shard_payload

# 3b) Run arbitrary SQL export
# sql_payload = session.export_sql(
#     sql='SELECT trip_mode, COUNT(*) AS n FROM v_trips GROUP BY 1 ORDER BY 2 DESC',
#     output_path=OUTPUT_DIR / 'trip_mode_counts.csv',
#     output_format='csv',
# )
# sql_payload

# 3c) Export transformed ActivitySim inputs (trips/persons)
# asim_payload = session.export_activitysim_inputs(
#     output_dir=OUTPUT_DIR / 'asim_inputs',
#     scenario_id='baseline-2030',
#     year=2030,
#     trips=TableTransformSpec(
#         columns=['person_id', 'trip_mode', 'depart', 'origin', 'destination'],
#         rename={'trip_mode': 'mode'},
#     ),
#     persons=TableTransformSpec(
#         columns=['person_id', 'household_id', 'value_of_time'],
#         rename={'value_of_time': 'vot'},
#     ),
# )
# asim_payload


## 4) Optional: Quick Validation

Load an exported file and inspect shape/columns before passing to downstream models.


In [ ]:
# example_path = OUTPUT_DIR / 'asim_inputs' / 'trips.csv'
# df = pd.read_csv(example_path)
# print(df.shape)
# display(df.head())
